Import pickle to load saved machine learning models.


In [ ]:
import pickle


Open and load the saved machine learning model.


In [ ]:
ml_model_file = open("ml_model.pkl", "rb")
ml_model = pickle.load(ml_model_file)
ml_model_file.close()


Import pandas to read and handle the dataset.


In [ ]:
import pandas as pd


Read the heart disease dataset.


In [ ]:
df = pd.read_csv("heart_dataset.csv")


Select all feature columns up to `thallium_heart_rate`.


In [ ]:
dataset_features = df.loc[:, :"thallium_heart_rate"]


Store the feature names in a list.


In [ ]:
feature_names = dataset_features.columns.to_list()


Get all decision trees from the random forest model.


In [ ]:
decision_trees = ml_model.estimators_


Print the number of decision trees.


In [ ]:
print(len(decision_trees))


Select the first decision tree for visualization.


In [ ]:
decision_tree = decision_trees[0]


Import `plot_tree` to visualize decision trees.


In [ ]:
from sklearn.tree import plot_tree


Plot the full decision tree.


In [ ]:
_ = plot_tree(decision_tree)


Plot only the first two levels of the decision tree with feature names.


In [ ]:
_ = plot_tree(
    decision_tree,
    max_depth=2,
    feature_names=feature_names,
    fontsize=6
)


Open and load the saved deep learning model.


In [ ]:
dl_model_file = open("dl_model.pkl", "rb")
dl_model = pickle.load(dl_model_file)
dl_model_file.close()


Convert the dataset features into a NumPy array.


In [ ]:
dataset_features = dataset_features.to_numpy()


Select the first sample from the dataset.


In [ ]:
one_sample_features = dataset_features[0]


Import PyTorch to convert NumPy arrays into tensors.


In [ ]:
import torch


Wrap the deep learning model so SHAP can call it using NumPy input.


In [ ]:
dl_model_wrapped = lambda features: dl_model(
    torch.Tensor(features)
).detach().numpy()


Import `tqdm` and SHAP for model explanation.


In [ ]:
import tqdm.auto
import shap


Create a SHAP KernelExplainer using the first 100 samples as background data.


In [ ]:
shap_explainer = shap.KernelExplainer(
    dl_model_wrapped,
    dataset_features[:100]
)


Get the expected/base prediction value from SHAP.


In [ ]:
shap_expected_value = shap_explainer.expected_value


Calculate SHAP values for one sample.


In [ ]:
one_sample_shap_values = shap_explainer.shap_values(one_sample_features)


Print the SHAP values for the selected sample.


In [ ]:
print(one_sample_shap_values)


Import Matplotlib to display SHAP plots.


In [ ]:
import matplotlib.pyplot as plt


Create a force plot to explain the prediction for one sample.


In [ ]:
shap.force_plot(
    shap_expected_value,
    one_sample_shap_values[0],
    one_sample_features,
    feature_names=feature_names,
    matplotlib=True,
    show=True
)


Select the first 10 samples for a summary explanation.


In [ ]:
multiple_samples_features = dataset_features[:10]


Calculate SHAP values for the first 10 samples.


In [ ]:
multiple_samples_shap_values = shap_explainer.shap_values(
    multiple_samples_features
)


Create a summary plot showing feature importance across multiple samples.


In [ ]:
shap.summary_plot(
    multiple_samples_shap_values,
    multiple_samples_features,
    feature_names=feature_names
)
